#### 데이터 전처리 

### 데이터 불러오기 


In [2]:
from pathlib import Path
import pandas as pd


# police.ipynb가 project_root에 있다고 가정

df_week = pd.read_excel('../data/police_week.xlsx')
df_week.head()

,범죄대분류,범죄중분류,일,월,화,수,목,금,토
0,강력범죄,살인기수,38,45,48,40,37,49,40
1,강력범죄,살인미수등,71,70,68,62,73,85,53
2,강력범죄,강도,111,113,116,105,124,121,108
3,강력범죄,강간,869,686,734,719,682,722,898
4,강력범죄,유사강간,145,99,109,92,89,112,136


### 필요없는 칼럼 삭제 

In [38]:
drop_targets = [
    "유사강간",
    "기타 강간/강제추행등",
    "방화",
    "체포/감금",
    "약취/유인",
    "폭력행위등",
    "공갈",
    "손괴",
]

# 컬럼명 정리
df_week.columns = df_week.columns.astype(str).str.strip()

# 1) 컬럼으로 존재하면 삭제
df_week = df_week.drop(columns=drop_targets, errors="ignore")

# 2) 범죄중분류 '행 값'으로 존재하면 해당 행 삭제
if "범죄중분류" in df_week.columns:
    df_week["범죄중분류"] = df_week["범죄중분류"].astype(str).str.strip()
    df_week = df_week[~df_week["범죄중분류"].isin(drop_targets)].copy()

display(df_week.head())
print(df_week.shape)

,범죄대분류,범죄중분류,일,월,화,수,목,금,토
0,강력범죄,살인기수,38,45,48,40,37,49,40
1,강력범죄,살인미수등,71,70,68,62,73,85,53
2,강력범죄,강도,111,113,116,105,124,121,108
3,강력범죄,강간,869,686,734,719,682,722,898
8,절도범죄,절도범죄,25466,26239,25671,26061,26098,28156,29266


(9, 9)


### 다른 파일과 연결을 위해 요일칼럼을 행으로 변환 

In [43]:
import pandas as pd

# 요일 컬럼을 행으로 변환
day_order = ["일", "월", "화", "수", "목", "금", "토"]
day_cols = [c for c in day_order if c in df_week.columns]

df_week_long = df_week.melt(
    id_vars=["범죄대분류", "범죄중분류"],
    value_vars=day_cols,
    var_name="요일",
    value_name="발생건수"
)

# 정리
df_week_long["발생건수"] = pd.to_numeric(df_week_long["발생건수"], errors="coerce").fillna(0).astype(int)
df_week_long["요일"] = pd.Categorical(df_week_long["요일"], categories=day_order, ordered=True)
df_week_long = df_week_long.sort_values(["범죄대분류", "범죄중분류", "요일"]).reset_index(drop=True)

display(df_week_long.head(20))
print(df_week_long.shape)


,범죄대분류,범죄중분류,요일,발생건수
0,강력범죄,강간,일,869
1,강력범죄,강간,월,686
2,강력범죄,강간,화,734
3,강력범죄,강간,수,719
4,강력범죄,강간,목,682
5,강력범죄,강간,금,722
6,강력범죄,강간,토,898
7,강력범죄,강도,일,111
8,강력범죄,강도,월,113
9,강력범죄,강도,화,116


(63, 4)


### csv 로 저장

In [44]:
df_week_long.to_csv("1_data/police_week_long_clean.csv", index=False, encoding="utf-8-sig")